# Code Distillation: Natural Language to Python

This customer-configured notebook validates the exact preserved data, submits or resumes one `GlobalStandard` supervised training job, and reports only service training metrics. It performs no deployment, fine-tuned inference or base-model comparison.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.


## 1. Configure and import

Install `cookbooks/requirements.lock`, authenticate with `az login` or another `DefaultAzureCredential` source, and copy this module's `.env.template` to ignored `.env`. Set your project endpoint and `FOUNDRY_MODEL`. The source model example is Alibaba `qwen3.8-27b`, expected version `1`; select a model available for fine-tuning in your own project. Dated model identifiers pin dated versions; an optional `FOUNDRY_MODEL_VERSION` checks returned metadata when supplied by the service, but does not pin an undated model alias.

Run from the module or `notebooks` directory. Blank optional polling settings use the documented defaults. The exact generated labels/traces are bundled, so no generation endpoint or teacher configuration is required.


In [ ]:
MODULE_NAME = '02-code-distillation'
HYPERPARAMETERS = {'n_epochs': 1, 'batch_size': 1, 'learning_rate_multiplier': 1.3}
TRAIN_FILE = 'training_data.jsonl'
VALIDATION_FILE = 'validation_data.jsonl'
TEST_FILE = None
REQUIRE_TOOLS = False
JOB_SUFFIX = '02-code-distillation'
EXPECTED = {'training_data.jsonl': {'rows': 1576, 'sha256': '18950b69bc0d63a4f56b7194b7eafa76b34aa6acecf2aff7321e18f488280708'}, 'validation_data.jsonl': {'rows': 83, 'sha256': '821a800a749279b540f9562e2ae86fca7649d088f5238f8a23256ec439b0c3cc'}}
CHAT_FILES = ['training_data.jsonl', 'validation_data.jsonl']

import csv
import hashlib
import json
import math
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if ROOT.name != MODULE_NAME or not (ROOT / "data").is_dir():
    raise RuntimeError(f"Run from {MODULE_NAME} or its notebooks directory.")
TRACK_ROOT = ROOT.parent
COOKBOOK_ROOT = TRACK_ROOT.parent
sys.path.insert(0, str(COOKBOOK_ROOT))
sys.path.insert(0, str(TRACK_ROOT))

from shared.auth import create_project_context
from shared.config import load_foundry_config
from stage05_runtime import monitor_fine_tuning_job, wait_for_file_processed

if (ROOT / ".env").is_file():
    load_dotenv(ROOT / ".env")

def setting(name, default=""):
    value = os.environ.get(name, "").strip()
    return value if value else default

def positive_seconds(name, default):
    value = float(setting(name, str(default)))
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be a finite positive number.")
    return value


MODEL = setting("FOUNDRY_MODEL")
EXPECTED_MODEL_VERSION = setting("FOUNDRY_MODEL_VERSION")
TRAINING_TYPE = setting("FOUNDRY_TRAINING_TYPE", "GlobalStandard")
if TRAINING_TYPE != "GlobalStandard":
    raise ValueError("This recipe requires FOUNDRY_TRAINING_TYPE=GlobalStandard.")
RUN_ID = setting("FOUNDRY_RUN_ID", "default")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_ID):
    raise ValueError("FOUNDRY_RUN_ID must be 1-64 letters, digits, underscores or hyphens.")
FILE_POLL_SECONDS = positive_seconds("FOUNDRY_FILE_POLL_SECONDS", 5)
JOB_POLL_SECONDS = positive_seconds("FOUNDRY_JOB_POLL_SECONDS", 60)
FILE_TIMEOUT_SECONDS = positive_seconds("FOUNDRY_FILE_TIMEOUT_SECONDS", 900)
JOB_TIMEOUT_SECONDS = positive_seconds("FOUNDRY_JOB_TIMEOUT_SECONDS", 86400)
RUN_DIR = ROOT / "outputs" / RUN_ID
STATE_PATH = RUN_DIR / "run-state.json"
DATA = ROOT / "data"
print({"model": MODEL or "not configured", "training_type": TRAINING_TYPE, "recipe": HYPERPARAMETERS})


## 2. Verify exact data and recipe

The preserved contract is the exact 1,576 training and 83 validation rows from `Demos/NL_to_Python_Distillation`. No examples are generated, filtered, judged or rewritten.

The SHA-256 guards stop on altered bytes. JSON/chat validation and exact split-overlap checks run locally without credentials or paid operations. Test/held-out data are never uploaded. See `data/README.md` for source lineage and preparation details.


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def canonical(row):
    return json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def read_contract(name, expected):
    path = DATA / name
    digest = sha256(path)
    if digest != expected["sha256"]:
        raise RuntimeError(f"{name}: SHA-256 mismatch; expected {expected['sha256']}, got {digest}.")
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(rows) != expected["rows"]:
        raise RuntimeError(f"{name}: expected {expected['rows']} rows, got {len(rows)}.")
    return rows

def validate_chat_rows(name, rows, require_tools=False):
    tool_turns = 0
    for row_number, row in enumerate(rows, 1):
        messages = row.get("messages")
        if not isinstance(messages, list) or not messages:
            raise ValueError(f"{name}:{row_number}: missing non-empty messages list.")
        row_tool_turns = 0
        pending = set()
        for message in messages:
            role = message.get("role")
            if role not in {"system", "user", "assistant", "tool"}:
                raise ValueError(f"{name}:{row_number}: invalid chat role {role!r}.")
            if "tool_calls" in message and message["tool_calls"] is None:
                raise ValueError(f"{name}:{row_number}: explicit null tool_calls.")
            calls = message.get("tool_calls", [])
            if role == "assistant" and calls:
                row_tool_turns += 1
                if "content" in message and not isinstance(message["content"], str):
                    raise ValueError(f"{name}:{row_number}: omit null assistant tool-call content.")
                if pending:
                    raise ValueError(f"{name}:{row_number}: new tool calls before pending replies.")
                for call in calls:
                    call_id = call.get("id")
                    function = call.get("function", {})
                    if not call_id or call_id in pending:
                        raise ValueError(f"{name}:{row_number}: missing or duplicate tool-call ID.")
                    if not function.get("name") or not isinstance(function.get("arguments"), str):
                        raise ValueError(f"{name}:{row_number}: invalid function name/arguments.")
                    json.loads(function["arguments"])
                    pending.add(call_id)
            else:
                if not isinstance(message.get("content"), str):
                    raise ValueError(f"{name}:{row_number}: message content must be text.")
                if role == "tool":
                    call_id = message.get("tool_call_id")
                    if call_id not in pending:
                        raise ValueError(f"{name}:{row_number}: unmatched tool response.")
                    pending.remove(call_id)
                elif pending:
                    raise ValueError(f"{name}:{row_number}: missing tool response before {role} turn.")
        # A final assistant tool call is a valid supervised prediction target.
        if pending and not (messages[-1]["role"] == "assistant" and messages[-1].get("tool_calls")):
            raise ValueError(f"{name}:{row_number}: incomplete tool replies.")
        if require_tools and not row_tool_turns:
            raise ValueError(f"{name}:{row_number}: no assistant tool-call supervision.")
        if not any(message["role"] == "assistant" for message in messages):
            raise ValueError(f"{name}:{row_number}: no assistant supervision.")
        tool_turns += row_tool_turns
    return tool_turns

datasets = {name: read_contract(name, expected) for name, expected in EXPECTED.items()}
lineage = {name: {"rows": len(rows), "sha256": sha256(DATA / name)}
           for name, rows in datasets.items()}
for name in CHAT_FILES:
    lineage[name]["assistant_tool_call_turns"] = validate_chat_rows(
        name, datasets[name], require_tools=REQUIRE_TOOLS)
split_names = [TRAIN_FILE, VALIDATION_FILE] + ([TEST_FILE] if TEST_FILE else [])
for index, left in enumerate(split_names):
    for right in split_names[index + 1:]:
        overlap = set(map(canonical, datasets[left])) & set(map(canonical, datasets[right]))
        if overlap:
            raise ValueError(f"{left} and {right} overlap in {len(overlap)} exact rows.")


print(json.dumps(lineage, indent=2))


## 3. Submission state and result parsing

Runtime evidence stays under ignored `outputs/<run-id>`. A blank `FOUNDRY_RUN_ID` uses `default`, so a rerun resumes the same job. Set a new run ID only for an intentional new paid run. A stored signature prevents reuse across project/model/data changes. Uploaded IDs are persisted before processing waits, and every resumed upload is checked again. An interrupted submission without a saved job ID requires explicit reconciliation with the service rather than a blind duplicate submission.

Each service metric column is kept separate; sampled and full validation metrics are not spliced into one misleading curve. Invalid/non-finite values and missing loss observations are explicit errors.


In [ ]:
def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def load_state(signature):
    if not STATE_PATH.exists():
        return {"signature": signature}
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    if state.get("signature") != signature:
        raise RuntimeError("This run ID belongs to different project/model/data settings. Choose a new FOUNDRY_RUN_ID.")
    if state.get("submission_pending") and not state.get("job_id"):
        raise RuntimeError(
            "A previous submission may have reached the service without a saved response. "
            "Inspect the project's jobs and reconcile run-state.json with its job_id before rerunning; "
            "do not submit another job blindly.")
    return state

def upload_once(client, state, name):
    key = name + "_file_id"
    if not state.get(key):
        with (DATA / name).open("rb") as handle:
            uploaded = client.files.create(file=(name, handle), purpose="fine-tune")
        if not uploaded.id:
            raise RuntimeError(f"Upload of {name} returned no file ID.")
        state[key] = uploaded.id
        write_json(STATE_PATH, state)
    return wait_for_file_processed(
        client, state[key], poll_interval_seconds=FILE_POLL_SECONDS,
        timeout_seconds=FILE_TIMEOUT_SECONDS)

def summarize_metrics(paths):
    series = {}
    for path in paths:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = [column for column in (reader.fieldnames or [])
                       if "loss" in column.casefold() or "token_accuracy" in column.casefold()]
            for row_number, row in enumerate(reader, 2):
                for column in columns:
                    raw = row.get(column)
                    if raw is None or not raw.strip():
                        continue
                    value = float(raw)
                    step = float(row["step"])
                    if not math.isfinite(value) or not math.isfinite(step):
                        raise ValueError(f"{path.name}:{row_number}: non-finite {column} or step.")
                    series.setdefault(column, []).append({"step": step, "value": value})
    if not any("loss" in column.casefold() and points for column, points in series.items()):
        raise RuntimeError("Service result CSVs contained no loss observations; no effectiveness conclusion can be drawn.")
    metrics = {}
    for column, points in series.items():
        values = [point["value"] for point in points]
        metrics[column] = {"observations": len(values), "first": values[0], "final": values[-1],
                           "minimum": min(values), "maximum": max(values),
                           "change": values[-1] - values[0] if len(values) > 1 else None,
                           "curve": points}
    return metrics


## 4. Opt in, upload, train, monitor and download

Executing this cell performs hosted calls and can incur training charges. The source hyperparameters above are retained (an empty dictionary keeps the source service-default recipe). The project client uses `DefaultAzureCredential`; uploads and fine-tuning use `AIProjectClient.get_openai_client()`. Failed/cancelled jobs and bounded wait timeouts raise clear errors. All clients close even on failure.

Creation POSTs disable SDK retries with `max_retries=0`, matching the shared duplicate-avoidance policy. A recorded pending job is monitored without another creation attempt; ambiguous submissions require reconciliation. The stored signature checks the requested project, model, data and recipe. Authentication and service errors propagate without fallback.


In [ ]:
if not MODEL:
    raise ValueError("Set FOUNDRY_MODEL to a fine-tuning model identifier available in your project.")
config = load_foundry_config(environ=dict(os.environ))
signature = {
    "project_endpoint": config.project_endpoint, "model": MODEL,
    "expected_model_version": EXPECTED_MODEL_VERSION,
    "training_type": TRAINING_TYPE, "hyperparameters": HYPERPARAMETERS,
    "data": {name: details["sha256"] for name, details in EXPECTED.items()},
}
state = load_state(signature)
RUN_DIR.mkdir(parents=True, exist_ok=True)
write_json(RUN_DIR / "data-lineage.json", lineage)

# All hosted resources are closed even if submission, polling or metrics fail.
with create_project_context(config) as context:
    client = context.openai_client
    if state.get("job_id"):
        job = client.fine_tuning.jobs.retrieve(state["job_id"])
        print(f"Resuming {job.id}; no new submission.")
    else:
        training = upload_once(client, state, TRAIN_FILE)
        validation = upload_once(client, state, VALIDATION_FILE)
        state["submission_pending"] = True
        write_json(STATE_PATH, state)
        submission_client = client.with_options(max_retries=0)
        job = submission_client.fine_tuning.jobs.create(
            model=MODEL, training_file=training.id, validation_file=validation.id,
            method={"type": "supervised",
                    "supervised": {"hyperparameters": HYPERPARAMETERS}},
            suffix=JOB_SUFFIX, extra_body={"trainingType": TRAINING_TYPE})
        if not job.id:
            raise RuntimeError("Fine-tuning submission returned no job ID; inspect the service before retrying.")
        state["job_id"] = job.id
        state["submission_pending"] = False
        write_json(STATE_PATH, state)
        print(f"Submitted {job.id}.")
    job = monitor_fine_tuning_job(
        client, job.id, poll_interval_seconds=JOB_POLL_SECONDS,
        timeout_seconds=JOB_TIMEOUT_SECONDS)
    resolved_version = (job.metadata or {}).get("model_version")
    if EXPECTED_MODEL_VERSION and resolved_version and str(resolved_version) != EXPECTED_MODEL_VERSION:
        raise RuntimeError(f"Service resolved model version {resolved_version}, expected {EXPECTED_MODEL_VERSION}.")
    write_json(RUN_DIR / "job.json", job.model_dump(mode="json"))
    if not job.result_files:
        raise RuntimeError("Succeeded job returned no result files; service loss metrics are unavailable.")
    result_paths = []
    for index, file_id in enumerate(job.result_files):
        response = client.files.content(file_id=file_id)
        content = response.read()
        if not isinstance(content, bytes):
            raise TypeError(f"Result file {file_id} did not return bytes.")
        path = RUN_DIR / f"result-{index}.csv"
        path.write_bytes(content)
        result_paths.append(path)
    metrics = summarize_metrics(result_paths)
    summary = {"status": str(job.status), "trained_tokens": job.trained_tokens,
               "lineage": lineage, "service_training_metrics": metrics}
    write_json(RUN_DIR / "training-summary.json", summary)
print(json.dumps(summary, indent=2))


## 5. Training-metrics-only result

Plot the fresh service loss columns and read the downloaded CSVs and `training-summary.json`. Token accuracy is included only when emitted by the service. Missing metrics are not inferred, and prior local runs are not used as evidence.


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots()
for column, details in metrics.items():
    if "loss" in column.casefold():
        points = details["curve"]
        axis.plot([point["step"] for point in points], [point["value"] for point in points],
                  marker="o", label=column)
axis.set(xlabel="Training step", ylabel="Service loss", title="Service training and validation loss")
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
figure.savefig(RUN_DIR / "service-loss.png", dpi=160)
plt.show()
plt.close(figure)
